# Session 2 · PyTorch: the 4 core components

PyTorch is the library most LLMs are built with. We only need 4 concepts:

| Concept | What it is, in one sentence |
|---|---|
| **Tensor** | A multi-dimensional array of numbers (like a NumPy array) that can also run on a GPU. |
| **Autograd** | PyTorch computes derivatives (gradients) automatically. |
| **`nn.Module`** | The base class for every piece of a neural network. |
| **Training loop** | 4 steps repeated over and over: forward → loss → backward → step. |

We will see all 4 in one example: we will train **the neuron from the slides**, $\hat{y} = w \cdot x + b$, to learn the relationship $y = 2x + 1$ from noisy data.

## Step 1 · Setup
The same imports as in Session 1:
- `torch.nn` (as `nn`) contains ready-made neural network building blocks, e.g. `nn.Linear` and `nn.MSELoss`.
- `torch.manual_seed(42)` makes random results reproducible.
- `device` tells us whether we are running on a GPU or a CPU. For this small example the CPU is enough.

In [ ]:
import math, os, time, random
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)

## Step 2 · Data (tensors) and model (`nn.Module`)

**The data are tensors.**
- `torch.linspace(-1, 1, 100)` creates 100 evenly spaced values of $x$ from −1 to 1.
- `.unsqueeze(1)` changes the shape from `(100,)` to `(100, 1)`: 100 examples with 1 feature each. This is the format `nn.Linear` expects.
- `Y = 2 * X + 1 + noise`: the "truth" we want the model to discover, with a little noise so it looks like real data.
- The **shape** is the most important property of a tensor. Half of all deep-learning bugs are wrong shapes.

**The model is an `nn.Module`.** Every piece of a neural network in PyTorch has the same structure:
- `__init__`: declares **what it is made of**. Here, one `nn.Linear(1, 1)`, which is exactly $\hat{y} = w \cdot x + b$, with $w$ and $b$ as **learnable parameters**.
- `forward`: says **what it does** with the input. We call it simply by writing `tiny(X)`.

`tiny.parameters()` shows $w$ and $b$: they start **random** and have `requires_grad=True`. This means PyTorch will track every operation on them so it can compute gradients.

In [ ]:
X = torch.linspace(-1, 1, 100).unsqueeze(1)          # (100, 1)
Y = 2 * X + 1 + 0.1 * torch.randn_like(X)           # (100, 1) με λίγο θόρυβο

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)                # y = w*x + b, με w και b να μαθαίνονται

    def forward(self, x):
        return self.linear(x)

tiny = TinyModel()
print(list(tiny.parameters()))

## Step 3 · Autograd and the training loop

Before the loop we create two tools:
- **`optimizer`**: the algorithm that corrects the parameters. `SGD` performs exactly the update from the slides, $w \leftarrow w - \eta \cdot \partial L / \partial w$, with learning rate $\eta$ = `lr=0.1`.
- **`loss_fn`**: how wrong the prediction is. `MSELoss` is the average of $(\hat{y} - y)^2$ over all examples.

**The 4 steps of the loop**, repeated 200 times:
1. **Forward:** `pred = tiny(X)`. The model predicts $\hat{y}$ for all 100 values of $x$ at once.
2. **Loss:** `loss = loss_fn(pred, Y)`. A single number that tells us how far off we are.
3. **Backward:** `loss.backward()`. This is where **autograd** works: using the chain rule, like the "mirror" in the slides, it computes $\partial L / \partial w$ and $\partial L / \partial b$ and stores them in the parameters' `.grad`.
4. **Step:** `optimizer.step()`. Each parameter takes a small step in the direction opposite to its gradient.

**Why `zero_grad()`?** PyTorch **adds** new gradients to the old ones. If we didn't clear them, every step would carry the gradients of all the previous steps.

**What to expect:** the loss drops quickly, and at the end $w \approx 2$ and $b \approx 1$. The model discovered the relationship $y = 2x + 1$ on its own, from the data alone.

> **Remember these 4 steps.** In Session 4 we will train our GPT with **exactly the same loop**. Only the model changes (a Transformer instead of `nn.Linear(1, 1)`) and the loss (cross-entropy instead of MSE).

In [ ]:
optimizer = torch.optim.SGD(tiny.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

for step in range(200):
    pred = tiny(X)                  # 1. forward: κάνε πρόβλεψη
    loss = loss_fn(pred, Y)         # 2. loss: πόσο λάθος είναι;
    optimizer.zero_grad()           #    (καθάρισε τα παλιά gradients)
    loss.backward()                 # 3. backward: υπολόγισε gradients
    optimizer.step()                # 4. step: διόρθωσε λίγο τις παραμέτρους
    if step % 40 == 0:
        print(f"step {step:3d} | loss {loss.item():.4f}")

print("w =", round(tiny.linear.weight.item(), 3), "| b =", round(tiny.linear.bias.item(), 3))